# Project Lab, Part 1: Funnels, Cohorts, RFM and a Sales Database

Notebook for **Chapter 15** of the *sql* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/sql/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: Funnel and Cohort-Retention Analysis for an E-Commerce Shop

In [ ]:
import sqlite3, random, sys
from datetime import date, datetime, timedelta
random.seed(42)
con = sqlite3.connect(sys.argv[1])
con.executescript("""
DROP TABLE IF EXISTS users; DROP TABLE IF EXISTS events; DROP TABLE IF EXISTS orders;
CREATE TABLE users  (user_id INTEGER PRIMARY KEY, signup_date TEXT NOT NULL, channel TEXT NOT NULL);
CREATE TABLE events (event_id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL, event TEXT NOT NULL, ts TEXT NOT NULL);
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL, order_date TEXT NOT NULL, amount REAL NOT NULL);
""")
start = date(2024, 1, 1); end = date(2024, 6, 30)
channels = [("search",.4,1.0),("social",.3,.7),("email",.2,1.3),("referral",.1,1.5)]  # name, share, quality
users=[]; events=[]; orders=[]
for uid in range(1, 3001):
    d = start + timedelta(days=random.randint(0, 150))
    r = random.random(); acc=0
    for ch,share,q in channels:
        acc+=share
        if r<=acc: break
    users.append((uid, d.isoformat(), ch))
    t = datetime(d.year,d.month,d.day,random.randint(8,22),random.randint(0,59))
    events.append((None,uid,'view',t.isoformat(' ')))
    if random.random() < .60*min(q,1.2):
        t += timedelta(minutes=random.randint(1,30)); events.append((None,uid,'cart',t.isoformat(' ')))
        if random.random() < .55:
            t += timedelta(minutes=random.randint(1,15)); events.append((None,uid,'checkout',t.isoformat(' ')))
            if random.random() < .70*min(q,1.1):
                t += timedelta(minutes=random.randint(1,10)); events.append((None,uid,'pay',t.isoformat(' ')))
                orders.append((None,uid,t.date().isoformat(),round(random.lognormvariate(3.9,.5),2)))
                p = .35*q                      # chance of buying again, shrinks each month
                for m in range(1,6):
                    if random.random() < p:
                        od = t.date()+timedelta(days=30*m+random.randint(-5,5))
                        if od<=end: orders.append((None,uid,od.isoformat(),round(random.lognormvariate(3.9,.5),2)))
                    p *= .75
con.executemany("INSERT INTO users VALUES (?,?,?)", users)
con.executemany("INSERT INTO events VALUES (?,?,?,?)", events)
con.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
con.commit()
for t in ("users","events","orders"):
    print(t, con.execute(f"select count(*) from {t}").fetchone()[0])

users 3000
events 6345
orders 1209


In [ ]:
import sqlite3, collections
con = sqlite3.connect("shop.db")
months = collections.defaultdict(set)
for u, d in con.execute("SELECT user_id, order_date FROM orders"):
    months[u].add(int(d[:4]) * 12 + int(d[5:7]))
cohort = collections.defaultdict(list)
for u, s in months.items():
    cohort[min(s)].append(u)
py = {c: (len(us), round(100 * sum(c + 1 in months[u] for u in us) / len(us), 1)) for c, us in cohort.items()}
rows = con.execute(open("sql/03_cohort.sql").read()).fetchall()
for label, size, m1, *_ in rows:
    y, m = map(int, label.split("-"))
    assert py[y * 12 + m] == (size, m1), label
print("all checks passed:", len(rows), "cohorts match")

all checks passed: 5 cohorts match


## 15.3 Project: RFM Customer Segmentation in SQL with a Written Report

## 15.4 Project: Design a Sales Database from Scratch, with 20 Business Queries

In [ ]:
import sqlite3, random, sys
from datetime import date, timedelta
random.seed(11)
con = sqlite3.connect(sys.argv[1]); con.execute("PRAGMA foreign_keys=ON")
con.executescript(open("sql/00_schema.sql").read())
regions=["North","South","East","West"]
con.executemany("INSERT INTO regions VALUES (?,?)",list(enumerate(regions,1)))
emps=[(1,"Asha Rao",1,None,"2019-04-01")]
names=["Ravi","Meera","Kiran","Sana","Dev","Isha","Omar","Tara"]
for i in range(2,10): emps.append((i,f"{names[i-2]} Nair",(i%4)+1,1,f"202{random.randint(0,3)}-0{random.randint(1,9)}-15"))
con.executemany("INSERT INTO employees VALUES (?,?,?,?,?)",emps)
con.executemany("INSERT INTO categories VALUES (?,?)",[(1,"Stationery"),(2,"Electronics"),(3,"Home"),(4,"Toys")])
prods=[]
for i in range(1,25):
    cat=(i-1)%4+1; price=round(random.uniform(40,900),2); cost=round(price*random.uniform(.5,.85),2)
    prods.append((i,f"SKU-{i:03d}",f"Product {i}",cat,price,cost))
con.executemany("INSERT INTO products VALUES (?,?,?,?,?,?)",prods)
con.executemany("INSERT INTO customers VALUES (?,?,?,?,?)",
  [(i,f"Customer {i}",f"c{i}@example.com",random.randint(1,4),(date(2023,1,1)+timedelta(days=random.randint(0,300))).isoformat()) for i in range(1,61)])
oid=0
for _ in range(240):
    oid+=1
    d=date(2024,1,1)+timedelta(days=random.randint(0,364))
    st=random.choices(["paid","shipped","cancelled","refunded"],[50,35,10,5])[0]
    con.execute("INSERT INTO orders VALUES (?,?,?,?,?)",(oid,random.randint(1,55),random.randint(2,9),d.isoformat(),st))
    for p in random.sample(range(1,25),random.randint(1,4)):
        con.execute("INSERT INTO order_items VALUES (?,?,?,?)",(oid,p,random.randint(1,5),prods[p-1][4]))
con.commit()
for t in ["regions","employees","customers","categories","products","orders","order_items"]:
    print(f"{t:12}", con.execute(f"select count(*) from {t}").fetchone()[0])

regions      4
employees    9
customers    60
categories   4
products     24
orders       240
order_items  606


## 15.5 Worked Extension: Compare Channels, Catch a Fan-Out and Test the Gap

In [ ]:
import math
# social: 96 buyers of 906 users.  search: 272 buyers of 1201 users.
x1, n1, x2, n2 = 96, 906, 272, 1201
p1, p2 = x1 / n1, x2 / n2
pool = (x1 + x2) / (n1 + n2)
se = math.sqrt(pool * (1 - pool) * (1 / n1 + 1 / n2))
z = (p2 - p1) / se
print("social %.1f%%  search %.1f%%  gap %.1f points" % (100 * p1, 100 * p2, 100 * (p2 - p1)))
print("z = %.1f" % z)
se_gap = math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
lo, hi = (p2 - p1) - 1.96 * se_gap, (p2 - p1) + 1.96 * se_gap
print("95%% interval for the gap: %.1f to %.1f points" % (100 * lo, 100 * hi))

social 10.6%  search 22.6%  gap 12.1 points
z = 7.2
95% interval for the gap: 9.0 to 15.2 points


## Chapter recap

## Practice set

Problem 15.P.1 (easy) Write the success criteria for the question "Which region has the most cancellations?" in two lines. **Answer:** Example: "A table of cancellation rate per region that reconciles to the total cancelled count; a note on regions with fewer than 30 orders." Any measurable, checkable sentence is correct.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/sql/ch15.html